# COLMBO-DF: Audio Deepfake Detection

این notebook مراحل کامل اجرای COLMBO-DF را در Google Colab Pro Plus پوشش می‌دهد.

**مراحل:**
1. اتصال به Google Drive و آپلود کد
2. نصب وابستگی‌ها
3. احراز هویت HuggingFace
4. آماده‌سازی دیتاست
5. تولید CoT annotations
6. آموزش مدل
7. ارزیابی

## مرحله ۱: اتصال به Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# مسیر پروژه در Drive را تنظیم کنید
PROJECT_DIR = '/content/drive/MyDrive/colmbo_df'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')

### آپلود فایل‌های پروژه

فایل‌های زیر را از کامپیوتر محلی به پوشه `colmbo_df` در Google Drive آپلود کنید:
- `config.py`
- `dataset.py`
- `evaluate.py`
- `features.py`
- `generate_cot.py`
- `inference.py`
- `model.py`
- `prepare_data.py`
- `train.py`
- `requirements.txt`

**یا** از گیت‌هاب کلون کنید (اگر ریپو دارید):
```
!git clone https://github.com/YOUR_USERNAME/colmbo_df.git /content/drive/MyDrive/colmbo_df
```

In [ ]:
# تغییر دایرکتوری به پروژه
%cd {PROJECT_DIR}
!ls -la

## مرحله ۲: بررسی GPU و نصب وابستگی‌ها

In [ ]:
# بررسی GPU
!nvidia-smi

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# نصب وابستگی‌ها
!pip install -q \
    'torch>=2.1.0' \
    'torchaudio>=2.1.0' \
    'transformers>=4.44.0' \
    'accelerate>=0.30.0' \
    'librosa>=0.10.0' \
    'praat-parselmouth>=0.4.3' \
    'soundfile>=0.12.1' \
    'numpy>=1.24.0' \
    'tqdm>=4.66.0' \
    'scikit-learn>=1.4.0'

print('Installation complete!')

## مرحله ۳: احراز هویت HuggingFace (برای Llama 3.2)

In [ ]:
# توکن را از https://huggingface.co/settings/tokens بگیرید
from huggingface_hub import login
from google.colab import userdata

# روش ۱: از Colab Secrets (توصیه شده - امن‌تر)
# در Colab: Tools > Secrets > Add new secret با نام HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab Secrets')
except Exception:
    # روش ۲: وارد کردن مستقیم (کمتر امن)
    HF_TOKEN = input('HuggingFace token را وارد کنید: ')
    login(token=HF_TOKEN)

## مرحله ۴: آماده‌سازی دیتاست ASVspoof 2019

### دانلود دیتاست
دیتاست ASVspoof 2019 LA را از سایت رسمی دانلود کنید:
- `https://datashare.ed.ac.uk/handle/10283/3336`

فایل‌های مورد نیاز:
- `LA.zip` (شامل train/dev/eval splits)

پس از دانلود، فایل را در Google Drive قرار دهید.

In [ ]:
# مسیر دیتاست را تنظیم کنید
ASVSPOOF_ROOT = '/content/drive/MyDrive/datasets/ASVspoof2019/LA'

# اگر فایل zip در Drive دارید، اینجا extract کنید:
# ZIP_PATH = '/content/drive/MyDrive/datasets/LA.zip'
# EXTRACT_DIR = '/content/drive/MyDrive/datasets/ASVspoof2019'
# !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}

import os
if os.path.exists(ASVSPOOF_ROOT):
    print('Dataset found!')
    !ls {ASVSPOOF_ROOT}
else:
    print(f'Dataset NOT found at {ASVSPOOF_ROOT}')
    print('Please upload the ASVspoof2019 LA dataset to your Google Drive')

In [ ]:
# ساختن پوشه data
!mkdir -p {PROJECT_DIR}/data

# اجرای prepare_data.py برای ساختن manifest های JSON
!python prepare_data.py \
    --asvspoof_root {ASVSPOOF_ROOT} \
    --output_train  data/fakereason_train.json \
    --output_eval   data/fakereason_eval.json

print('Data preparation complete!')

## مرحله ۵: تولید CoT Annotations (اختیاری)

In [ ]:
# برای تولید CoT به یک LLM قوی‌تر (مثل Qwen3-30B) نیاز دارید
# اگر API دارید:

# !python generate_cot.py \
#     --manifest data/fakereason_train.json \
#     --output   data/fakereason_train_cot.json \
#     --model    Qwen/Qwen3-30B-A3B-Instruct \
#     --short

print('اگر از mode=shortcot یا nocot استفاده می‌کنید، این مرحله اختیاری است')

## مرحله ۶: آموزش مدل

In [ ]:
# تنظیمات آموزش
# در Colab Pro Plus با A100 (40GB VRAM):
# - batch_size=4 و grad_accum=4 کار می‌کند
# - bf16=True برای سرعت بیشتر

!python train.py \
    --manifest_train data/fakereason_train.json \
    --manifest_eval  data/fakereason_eval.json \
    --output_dir     checkpoints/shortcot \
    --mode           shortcot \
    --encoder_name   microsoft/wavlm-base-plus \
    --llm_name       meta-llama/Llama-3.2-1B-Instruct \
    --batch_size     4 \
    --grad_accum     4 \
    --epochs         3 \
    --lr             1e-4

In [ ]:
# اگر می‌خواهید مقایسه mode های مختلف (مثل مقاله):

# NoCoT mode:
# !python train.py --mode nocot --output_dir checkpoints/nocot ...

# Full CoT mode:
# !python train.py --mode cot --output_dir checkpoints/cot ...

## مرحله ۷: ارزیابی مدل

In [ ]:
# ارزیابی با آخرین checkpoint
import glob
checkpoints = sorted(glob.glob('checkpoints/shortcot/checkpoint-*'))
if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f'Latest checkpoint: {latest_ckpt}')
    
    !python evaluate.py \
        --manifest   data/fakereason_eval.json \
        --checkpoint {latest_ckpt} \
        --output     results/shortcot_eval.json
else:
    print('No checkpoints found. Run training first.')

## نکات مهم

### ذخیره‌سازی
- Checkpoints را در Google Drive ذخیره کنید تا بعد از قطع session از بین نروند
- `checkpoints/` را به یک مسیر داخل Drive ببندید

### حافظه GPU
- اگر OOM خطا گرفتید:
  - `batch_size` را به `2` کاهش دهید
  - `grad_accum` را به `8` افزایش دهید (effective batch همان می‌ماند)
  - `max_audio_len=48000` (3 ثانیه به جای 5)

### زمان Session
- Colab Pro Plus تا 24 ساعت session می‌دهد
- برای آموزش طولانی از `--resume` برای ادامه از checkpoint استفاده کنید